# Recursive MHM in identical leaf spaces

This notebook consumes the ten current Cartesian Q2/P1 acquisitions: five outer meshes and both zero and affine weak Dirichlet pressure. Each outer macrocell has four inner macrocells, each with four fine Q2 quadrilaterals. Recursive and flat MHM share these leaf and trace spaces. The reference independently assembles the complete original system with FEniCS Basix 0.9.0; its attributed SciPy solver arithmetic is shared.

The measured smooth-case rates and finite rank/gauge controls do not establish a mesh-uniform inf-sup bound or transfer a single-cell Pk theorem to Q2 on a fine submesh. Numerical bases, orientation maps and evaluation tables are part of each archive. This notebook replays the selected saved field without solving or tabulating a new basis.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
from IPython.display import Image, display

root = Path.cwd()
if not (root / "examples").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from examples.nested_field_archive import display_fields, read_archive

record_path = root / "examples/results/nested.json"
record = json.loads(record_path.read_text())
assert record["schema"] == "pymhm-recursive-study-v2"
assert len(record["rows"]) == 10
assert {(row["n"], row["boundary_case"]) for row in record["rows"]} == {
    (n, boundary) for n in (1, 2, 4, 8, 16)
    for boundary in ("sine_zero_dirichlet", "affine_plus_sine")
}
table = [
    {key: row[key] for key in (
        "n", "boundary_case", "outer_global_dofs", "flat_global_dofs", "pressure_l2", "flux_l2"
    )}
    for row in record["rows"]
]
table


In [ ]:
assert record["maximum_original_source_and_constraint_relative_residual"] < 1e-10
assert record["maximum_recursive_flat_pressure_and_raw_flux_norm_difference"] < 1e-9
assert record["maximum_norm_q8_q10_relative_sensitivity"] < 1e-9
assert max(row["relative_leaf_difference"] for row in record["rows"]) < 1e-11

native_path = record_path.parent / record["native_verification"]["record"]
assert hashlib.sha256(native_path.read_bytes()).hexdigest() == record["native_verification"]["record_sha256"]
native = json.loads(native_path.read_text())
assert native["complete"] and native["cases"] == 10
assert native["acquisition_source_identity_sha256"] == record["source_identity_sha256"]
assert native["maximum_native_original_relative"] < native["criteria"]["original"]
assert native["maximum_candidate_in_native_original_relative"] < native["criteria"]["original"]
assert native["maximum_same_space_physical_field_relative"] < native["criteria"]["physical_fields"]
assert native["maximum_represented_operator_relative_difference"] < native["criteria"]["represented_operator_identity"]
assert native["maximum_native_norm_q8_q10_relative_sensitivity"] < native["criteria"]["norm_integration"]
assert native["bitwise_data_only_replay_and_comparison_threads_1_2"]
{
    "observed_final_rates": {boundary: rates[-1] for boundary, rates in record["observed_rates"].items()},
    "native_original": native["maximum_native_original_relative"],
    "candidate_in_native_original": native["maximum_candidate_in_native_original_relative"],
    "physical_field_difference": native["maximum_same_space_physical_field_relative"],
    "reference": native["reference"],
}


In [ ]:
selected = record["display_field"]
archive = record_path.parent / selected["archive"]
assert hashlib.sha256(archive.read_bytes()).hexdigest() == selected["archive_sha256"]
acquisition, arrays = read_archive(archive)
assert acquisition["n"] == selected["n"] == 4
assert acquisition["boundary_case"] == selected["boundary_case"] == "sine_zero_dirichlet"
points, pressure, raw_flux = display_fields(arrays)
assert points.shape == (64, 289, 2)
assert pressure.shape == (64, 289)
assert raw_flux.shape == (64, 289, 2)
assert all(np.all(np.isfinite(value)) for value in (points, pressure, raw_flux))
{
    "acquisition_uuid": acquisition["acquisition_uuid"],
    "saved_one_sided_samples": pressure.size,
    "coefficient_precision_bits": acquisition["coefficient_precision_bits"],
    "actual_display_table_digest": acquisition["arrays_sha256"]["display_basis"],
    "pressure_range": (float(pressure.min()), float(pressure.max())),
}


The left convergence panel measures absolute analytical L2 errors. The right panel compares recursive and flat physical fields, independently of coefficient differences. The final pressure and raw-flux orders approach three and two in both boundary problems.

The spatial panels use the selected homogeneous n=4 archive. Dark lines mark actual outer macrofaces and thin lines mark actual inner macrofaces on analytical, numerical and error panels. Raw Darcy flux is minus the one-sided pressure gradient; it is distinct from an H(div) field and from the oriented normal-flux multiplier. Nine independent colorbars retain each panel's scale, including its pointwise error. Full broken H1 norms in the native record include both pressure L2 and gradient L2 terms.


In [ ]:
figure_folder = root / "docs/figures/nested"
provenance = json.loads((figure_folder / "provenance.json").read_text())
assert provenance["acquisition_uuid"] == acquisition["acquisition_uuid"]
assert provenance["input_archive_sha256"] == selected["archive_sha256"]
assert provenance["input_record_sha256"] == hashlib.sha256(record_path.read_bytes()).hexdigest()
for name in ("convergence", "fields"):
    figure = figure_folder / f"{name}.png"
    assert hashlib.sha256(figure.read_bytes()).hexdigest() == provenance["figures_sha256"][figure.name]
    display(Image(filename=str(figure), width=960))


The current analytical geometry is a declared equivalence study, distinct from a literal historical mesh or figure reproduction. Normal macroface equilibrium does not establish fine-cell conservation of the raw gradient. Reduced outer unknown counts measure condensation; setup, archival and verification costs do not demonstrate a runtime speedup.

See `docs/cases/nested.md` for the physical equations, finite rank/gauge argument, complete original-system criteria, source attribution and commands for a fresh acquisition and data-only rendering. External reference sources remain outside the versioned repository and release artifacts.
